# MVA v3 · NB04A-S2 — artifact-aware exploratory segmental BAF scan
**v2.0.0 · supersedes NB04A-S v1.0.0 · post-hoc exploratory**

## Why S1 is superseded

NB04A-S v1.0.0 produced 87/558 primary outlier windows and `robust_z` values in the thousands to ~80,000.
That is a calibration failure: the robust MAD scale collapsed toward zero, so a Normal-tail p-value was invalid.

Many apparent segments also overlapped centromeric/pericentromeric or chromosome-end regions.

**The 18 S1 segments are therefore not biological calls and must not be reported as mCA.**

## New question

After masking assembly-gap/centromere/telomere/heterochromatin regions, does any 5-Mb interval show a
**cross-validated improvement of a symmetric allelic-imbalance model over a centered beta-binomial noise model**
that is extreme relative to windows on other chromosomes?

## Frozen S2 rules before viewing S2 results

1. all autosomes scanned identically;
2. 5-Mb non-overlapping windows;
3. >=300 HQ heterozygous SNPs/window;
4. UCSC hg38 `gap` features of type `centromere`, `telomere`, or `heterochromatin`, plus **2 Mb flank**, exclude a window;
5. deterministic 2-fold genomic-marker cross-validation;
6. null = beta-binomial centered at BAF 0.5 with fitted overdispersion;
7. alternative = symmetric two-component beta-binomial at `0.5±delta`, fitting both `rho` and `delta`;
8. score = held-out log-likelihood improvement per marker;
9. empirical p-value = rank against eligible windows on **other chromosomes**; no Normal approximation;
10. primary window: empirical `p<=0.01`, BH `q<=0.05`, positive held-out score;
11. segment: primary window plus an immediately adjacent eligible window with empirical `p<=0.05`
    and positive held-out score;
12. variant-site DP is secondary only and cannot create a call.

This remains exploratory and cannot retroactively change the negative whole-chromosome result.

In [ ]:
import os
import importlib.util, subprocess, sys
for pkg in ["scipy","pandas","numpy","requests"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])

import numpy as np, pandas as pd, json, math, requests, io, gzip
from scipy.optimize import minimize, minimize_scalar
from scipy.special import betaln, gammaln, logsumexp
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive",force_remount=False)

ROOT = Path(os.environ.get("MVA_ROOT", "/path/to/MVA_Hackathon_2026"))
PRIVATE_IN=ROOT/"02_RESULTS_PRIVATE"/"NB04A_VALIDATION_v1_0_0"/"raw_het_snp_counts_private.parquet"
PRIVATE_OUT=ROOT/"02_RESULTS_PRIVATE"/"NB04A_SEGMENTAL_S2_v2_0_0"
PUBLIC_OUT=ROOT/"03_PUBLIC_ARTIFACTS"
PRIVATE_OUT.mkdir(parents=True,exist_ok=True)
PUBLIC_OUT.mkdir(parents=True,exist_ok=True)

assert PRIVATE_IN.exists(), f"Missing prerequisite: {PRIVATE_IN}"
raw=pd.read_parquet(PRIVATE_IN)
print("Private marker table loaded:",len(raw))

## 1. Primary marker QC

In [ ]:
z=raw[
    (raw.gq>=30) &
    (raw.dp>=10) & (raw.dp<=100) &
    (raw.refc>=5) & (raw.altc>=5)
].copy()

z=z.sort_values(["chrom","pos"])
z["kb"]=(z.pos//1000).astype(int)
z=z.drop_duplicates(["chrom","kb"],keep="first")
z["window_index"]=(z.pos//5_000_000).astype(int)
z["window_start"]=z.window_index*5_000_000
z["window_end"]=z.window_start+5_000_000-1

print("Markers after QC/thinning:",len(z))

## 2. UCSC hg38 artifact mask

In [ ]:
GAP_URL="https://hgdownload.soe.ucsc.edu/goldenPath/hg38/database/gap.txt.gz"
r=requests.get(GAP_URL,timeout=90)
r.raise_for_status()

# UCSC gap schema: bin, chrom, chromStart, chromEnd, ix, n, size, type, bridge
gap=pd.read_csv(
    io.BytesIO(gzip.decompress(r.content)),
    sep="\t",header=None,
    names=["bin","chrom","start","end","ix","n","size","type","bridge"]
)
gap["chrom"]=gap.chrom.astype(str).str.replace("^chr","",regex=True)
mask=gap[
    gap.chrom.isin([str(i) for i in range(1,23)]) &
    gap.type.isin(["centromere","telomere","heterochromatin"])
].copy()

FLANK=2_000_000
mask["start_ext"]=(mask.start-FLANK).clip(lower=0)
mask["end_ext"]=mask.end+FLANK

mask_intervals={
    c:list(g[["start_ext","end_ext"]].itertuples(index=False,name=None))
    for c,g in mask.groupby("chrom")
}

def overlaps_mask(chrom,start,end):
    for a,b in mask_intervals.get(str(chrom),[]):
        if start < b and end >= a:
            return True
    return False

print("Artifact-mask intervals:",len(mask))
print(mask.groupby("type").size().to_string())

## 3. Beta-binomial models and deterministic cross-validation

In [ ]:
def log_bb(k,n,p,rho):
    rho=float(np.clip(rho,1e-6,0.20))
    p=float(np.clip(p,1e-6,1-1e-6))
    conc=1/rho-1
    a=p*conc; b=(1-p)*conc
    return (
        gammaln(n+1)-gammaln(k+1)-gammaln(n-k+1)
        + betaln(k+a,n-k+b)-betaln(a,b)
    )

def fit_null(df):
    k=df.altc.to_numpy(int); n=df.n.to_numpy(int)
    res=minimize_scalar(
        lambda rho:-np.sum(log_bb(k,n,0.5,rho)),
        bounds=(1e-5,0.15),method="bounded",
        options={"xatol":1e-5}
    )
    return float(res.x)

def alt_nll(params,k,n):
    rho,delta=params
    p1=0.5-delta; p2=0.5+delta
    a=log_bb(k,n,p1,rho)
    b=log_bb(k,n,p2,rho)
    return -float(np.sum(logsumexp(np.vstack([a,b]),axis=0)-np.log(2.0)))

def fit_alt(df,rho_start=None):
    k=df.altc.to_numpy(int); n=df.n.to_numpy(int)
    if rho_start is None:
        rho_start=fit_null(df)
    starts=[
        [min(max(rho_start,1e-4),0.10),0.010],
        [min(max(rho_start/2,1e-4),0.10),0.030],
        [min(max(rho_start,1e-4),0.10),0.060],
    ]
    best=None
    for x0 in starts:
        res=minimize(
            alt_nll,x0,args=(k,n),method="L-BFGS-B",
            bounds=[(1e-5,0.15),(0.0,0.15)]
        )
        if best is None or res.fun < best.fun:
            best=res
    return float(best.x[0]),float(best.x[1])

def eval_null(df,rho):
    return float(np.sum(log_bb(
        df.altc.to_numpy(int),df.n.to_numpy(int),0.5,rho
    )))

def eval_alt(df,rho,delta):
    k=df.altc.to_numpy(int); n=df.n.to_numpy(int)
    a=log_bb(k,n,0.5-delta,rho)
    b=log_bb(k,n,0.5+delta,rho)
    return float(np.sum(logsumexp(np.vstack([a,b]),axis=0)-np.log(2.0)))

def cv_score(df):
    g=df.sort_values("pos").reset_index(drop=True)
    gains=[]; deltas=[]
    for parity in [0,1]:
        train=g.iloc[np.arange(len(g))%2==parity]
        test =g.iloc[np.arange(len(g))%2!=parity]
        if len(train)<120 or len(test)<120:
            return np.nan,np.nan
        rho0=fit_null(train)
        rho1,delta=fit_alt(train,rho0)
        ll0=eval_null(test,rho0)
        ll1=eval_alt(test,rho1,delta)
        gains.append((ll1-ll0)/len(test))
        deltas.append(delta)
    return float(np.mean(gains)),float(np.mean(deltas))

## 4. Scan all eligible unmasked windows

In [ ]:
rows=[]
groups=list(z.groupby(["chrom","window_index","window_start","window_end"]))
for ix,((chrom,wi,ws,we),g) in enumerate(groups,1):
    if len(g)<300:
        continue
    masked=overlaps_mask(chrom,int(ws),int(we))
    if masked:
        continue
    score,delta=cv_score(g)
    if not np.isfinite(score):
        continue
    rows.append({
        "chrom":str(chrom),
        "window_index":int(wi),
        "start":int(ws),
        "end":int(we),
        "n_markers":int(len(g)),
        "median_variant_dp":float(g.dp.median()),
        "cv_ll_gain_per_marker":score,
        "cv_delta":delta
    })
    if len(rows)%50==0:
        print("eligible windows scored:",len(rows))

w=pd.DataFrame(rows)
assert len(w)>100, "Too few eligible windows after masking."
print("Final eligible unmasked windows:",len(w))
display(w.sort_values("cv_ll_gain_per_marker",ascending=False).head(20))

## 5. Empirical leave-one-chromosome-out p-values + BH FDR

In [ ]:
def empirical_p(row,table):
    bg=table[table.chrom!=row.chrom].cv_ll_gain_per_marker.to_numpy(float)
    # Conservative +1 correction.
    return float((1+np.sum(bg>=row.cv_ll_gain_per_marker))/(1+len(bg)))

w["p_emp"]=w.apply(lambda r:empirical_p(r,w),axis=1)

def bh_qvalues(p):
    p=np.asarray(p,float)
    n=len(p)
    order=np.argsort(p)
    ranked=p[order]
    q=ranked*n/np.arange(1,n+1)
    q=np.minimum.accumulate(q[::-1])[::-1]
    out=np.empty(n,float)
    out[order]=np.minimum(q,1.0)
    return out

w["q_bh"]=bh_qvalues(w.p_emp.to_numpy())
w["primary_sig"]=(w.cv_ll_gain_per_marker>0)&(w.p_emp<=0.01)&(w.q_bh<=0.05)
w["support_sig"]=(w.cv_ll_gain_per_marker>0)&(w.p_emp<=0.05)

print("Primary windows:",int(w.primary_sig.sum()))
print("Empirical p minimum:",float(w.p_emp.min()))
print("BH q minimum:",float(w.q_bh.min()))

## 6. Spatial adjacency and segment construction

In [ ]:
w=w.sort_values(["chrom","window_index"]).reset_index(drop=True)
w["segment_support"]=False

for i,r in w.iterrows():
    if not bool(r.primary_sig):
        continue
    for j in [i-1,i+1]:
        if 0<=j<len(w):
            s=w.iloc[j]
            if (
                s.chrom==r.chrom and
                abs(int(s.window_index)-int(r.window_index))==1 and
                bool(s.support_sig)
            ):
                w.loc[i,"segment_support"]=True
                w.loc[j,"segment_support"]=True

segments=[]
for chrom,g in w[w.segment_support].groupby("chrom"):
    idxs=sorted(g.window_index.astype(int).tolist())
    if not idxs: continue
    comps=[[idxs[0]]]
    for x in idxs[1:]:
        if x==comps[-1][-1]+1:
            comps[-1].append(x)
        else:
            comps.append([x])
    for comp in comps:
        sg=w[(w.chrom==chrom)&(w.window_index.isin(comp))]
        if not sg.primary_sig.any():
            continue
        segments.append({
            "chrom":chrom,
            "start":int(sg.start.min()),
            "end":int(sg.end.max()),
            "n_windows":int(len(sg)),
            "best_empirical_p":float(sg.p_emp.min()),
            "best_q":float(sg.q_bh.min()),
            "median_cv_gain":float(sg.cv_ll_gain_per_marker.median()),
            "median_cv_delta":float(sg.cv_delta.median()),
            "median_variant_dp_ratio":np.nan
        })

seg=pd.DataFrame(segments)
print("S2 segment candidates:",len(seg))
display(seg)

## 7. Secondary variant-site DP and public handoff

In [ ]:
from datetime import datetime, timezone

global_dp=float(w.median_variant_dp.median())
w["variant_dp_ratio"]=w.median_variant_dp/global_dp

if len(seg):
    for i,r in seg.iterrows():
        sg=w[(w.chrom==r.chrom)&(w.start>=r.start)&(w.end<=r.end)]
        seg.loc[i,"median_variant_dp_ratio"]=float(sg.variant_dp_ratio.median())

# Private full window audit.
w.to_parquet(PRIVATE_OUT/"S2_window_results_private.parquet",index=False)

# Public final segments only.
cols=[
    "chrom","start","end","n_windows","best_empirical_p","best_q",
    "median_cv_gain","median_cv_delta","median_variant_dp_ratio"
]
if len(seg):
    pubseg=seg[cols].copy()
else:
    pubseg=pd.DataFrame(columns=cols)

pubseg.to_csv(PUBLIC_OUT/"NB04A_SEGMENTAL_S2_CANDIDATES.csv",index=False)

handoff={
    "schema":"MVA-NB04A-SEGMENTAL-S2-2.0",
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "status":"COMPLETE",
    "supersedes":"NB04A-SEGMENTAL-EXPLORATORY-1.0",
    "reason_for_supersession":"S1 robust-z null was miscalibrated (near-zero MAD; z in thousands to ~80,000) and did not mask major assembly/repetitive regions.",
    "analysis_type":"post-hoc exploratory genome-wide segmental scan",
    "artifact_mask":"UCSC hg38 gap centromere/telomere/heterochromatin +2 Mb flank",
    "model_null":"centered beta-binomial with fitted rho",
    "model_alt":"symmetric two-component beta-binomial 0.5±delta with fitted rho/delta",
    "score":"deterministic 2-fold held-out log-likelihood gain per marker",
    "significance":"leave-one-chromosome-out empirical p; BH q<=0.05",
    "primary_rule":"score>0, empirical p<=0.01, q<=0.05",
    "adjacency_rule":"immediately adjacent eligible window with score>0 and empirical p<=0.05",
    "n_eligible_unmasked_windows":int(len(w)),
    "n_primary_windows":int(w.primary_sig.sum()),
    "n_segment_candidates":int(len(pubseg)),
    "segment_candidates":pubseg.to_dict(orient="records"),
    "interpretation":{
        "positive":"exploratory non-artifact-masked segmental BAF signal; requires orthogonal CN/phase/cytogenetic validation",
        "negative":"no segment survives cross-validated empirical-null plus adjacency criteria"
    },
    "whole_chromosome_result_remains":"NOT_VALIDATED_WHOLE_CHROMOSOME_MCA",
    "limitations":[
        "post-hoc exploratory",
        "single bulk blood WGS",
        "variant-only VCF",
        "variant-site DP secondary only",
        "single-sample empirical background",
        "cannot quantify total cell-to-cell variegated aneuploidy"
    ],
    "privacy":{"full_window_table_public":False,"raw_genotypes_public":False}
}
(PUBLIC_OUT/"NB04A_SEGMENTAL_S2_HANDOFF.json").write_text(json.dumps(handoff,indent=2))
print(json.dumps(handoff,indent=2))